# Study 2 Part B on Google Colab

Runs the locked Part B of `docs/PREREGISTRATION_LEAKAGE.md` (Amendment 3): DeBERTa-v3-small and TF-IDF on row vs randomized group splits, plus scoring of released detectors.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU.*
2. Optional, for Meta Prompt Guard 2: accept its license on its Hugging Face page, create a read token, and add it in Colab's 🔑 *Secrets* panel as `HF_TOKEN`. Without it, that detector is recorded as unavailable and everything else still runs.
3. If the GitHub repo is private, add a GitHub token (read access to the repo) as the secret `GITHUB_TOKEN`.

Expected time: about 3–4 hours. Progress is saved to Google Drive after every fit. If Colab disconnects, run all cells again and it continues where it stopped.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/injection_lab_partb'

In [ ]:
import os
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

os.environ['HF_TOKEN'] = secret('HF_TOKEN') or ''
gh = secret('GITHUB_TOKEN')
url = f'https://{gh}@github.com/dannyg26/prompt_injection' if gh else 'https://github.com/dannyg26/prompt_injection'
if not os.path.exists('/content/prompt_injection'):
    os.system(f'git clone -q -b claude/vigilant-fermi-r2ovyf {url} /content/prompt_injection')
os.chdir('/content/prompt_injection')
!git log --oneline -1

In [ ]:
# Pinned scientific stack plus transformer dependencies. Running the study in a
# subprocess (next cell) means no runtime restart is needed after this install.
!pip install -q -r requirements-lock.txt
!pip install -q transformers==4.56.2 sentencepiece tiktoken protobuf huggingface_hub
!pip install -q --no-deps -e .

In [ ]:
!PYTHONPATH=src python scripts/run_partb.py --out "$OUT" 2>&1 | tail -n 60

## After it finishes

Download these two files and send them back (or commit them to `results/leakage/partb/` on the branch):
- `MyDrive/injection_lab_partb/partb_results.json`
- `MyDrive/injection_lab_partb/revisions.json`

The per-fit files in `tasks/` can be zipped and kept for the audit trail.

In [ ]:
from google.colab import files
files.download(f'{OUT}/partb_results.json')
files.download(f'{OUT}/revisions.json')